## Middleware

In [2]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

### Summarization middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older
context. Summarization is useful for the following:

• Long-running conversations that exceed context windows.

• Multi-turn dialogues with extensive history.

• Applications where preserving full conversation context matters.

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

# Message Summarization

agent = create_agent(
    model=ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=os.getenv("GEMINI_API_KEY")
    ),
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=ChatGoogleGenerativeAI(
                model="gemini-2.5-flash",
                google_api_key=os.getenv("GEMINI_API_KEY")
            ),
            trigger = ("messages",10),
            keep = ("messages",4)
        )
        ]
)

In [4]:
config = {"configurable":{"thread_id":"test-1"}}

In [5]:
questions = [
    "What is 2+2?",
    "what is 3+3?",
    "what is 4+4?",
    "what is 5+5?",
    "what is 6+6?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='28a5f1b0-7777-4a41-a77a-05877a0821ec'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a060dd-5875-7c40-bf15-78f6777c815b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 29, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 22}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='28a5f1b0-7777-4a41-a77a-05877a0821ec'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a060dd-5875-7c40-bf15-78f6777c815b-0', tool_c

### Human in loop Middleware

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    # Simulate reading an email by returning a dummy email content
    return f"Email content for {email_id}"

def send_email_tool(email_id: str, content: str) -> str:
    # Simulate sending an email by returning a confirmation message
    return f"Email sent to {email_id} with content: {content}"

In [7]:
agent = create_agent(
    model=ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=os.getenv("GEMINI_API_KEY")
    ),
    checkpointer = InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "reject","edit"],
                },
                read_email_tool:False ,
            }
        )
    ]
)

In [24]:
config = {"configurable":{"thread_id":"test-approve"}}

# Step 1 - Request

result = agent.invoke(
    {"messages":[HumanMessage(content="Send Email to john@test.com with subject 'Hello' and the body 'How are you?  '")]},config = config
)

In [25]:
result

{'messages': [HumanMessage(content="Send Email to john@test.com with subject 'Hello' and the body 'How are you?  '", additional_kwargs={}, response_metadata={}, id='50e72cc5-6723-4e53-9e87-689ed28bb12d'),
  AIMessage(content="Okay, I'll send an email to `john@test.com` with:\n\n*   **Subject:** Hello\n*   **Body:** How are you?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a060e5-c8f1-76e1-b901-022a708b2c4d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 144, 'total_tokens': 168, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 109}}),
  HumanMessage(content="Send Email to john@test.com with subject 'Hello' and the body 'How are you?  '", additional_kwargs={}, response_metadata={}, id='7784fe60-8847-4325-8f1f-bfde41ba5b44'),
  AIMessage(content="Okay, I'll send an email to `john@test.

In [26]:
# Step 2 - Approve

from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused ! Approving the request...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type","approve"}
                ]
            }
        ),
        config = config
    )
    print(f"Result : {result['messages'][-1].content}")

In [27]:
result

{'messages': [HumanMessage(content="Send Email to john@test.com with subject 'Hello' and the body 'How are you?  '", additional_kwargs={}, response_metadata={}, id='50e72cc5-6723-4e53-9e87-689ed28bb12d'),
  AIMessage(content="Okay, I'll send an email to `john@test.com` with:\n\n*   **Subject:** Hello\n*   **Body:** How are you?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a060e5-c8f1-76e1-b901-022a708b2c4d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 144, 'total_tokens': 168, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 109}}),
  HumanMessage(content="Send Email to john@test.com with subject 'Hello' and the body 'How are you?  '", additional_kwargs={}, response_metadata={}, id='7784fe60-8847-4325-8f1f-bfde41ba5b44'),
  AIMessage(content="Okay, I'll send an email to `john@test.

In [28]:
# Step 3 - Reject

config = {"configurable":{"thread_id":"test-reject"}}

# Step 1 - Request

result = agent.invoke(
    {"messages":[HumanMessage(content="Send Email to john@test.com with subject 'Hello' and the body 'How are you?  '")]},config
)

In [29]:
# Step 2 - Approve

from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused ! Rejecting the request...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type","reject"}
                ]
            }
        ),
        config = config
    )
    print(f"Result : {result['messages'][-1].content}")

In [30]:
result

{'messages': [HumanMessage(content="Send Email to john@test.com with subject 'Hello' and the body 'How are you?  '", additional_kwargs={}, response_metadata={}, id='5bc91ba5-b287-4740-b9a8-43ccd578005d'),
  AIMessage(content='To send an email programmatically, you\'ll need an email client or a script that can interact with an SMTP (Simple Mail Transfer Protocol) server.\n\nHere are a few common ways:\n\n---\n\n### 1. Using Python (Recommended for most applications)\n\nThis method is robust and works across different operating systems. You\'ll need an SMTP server (like Gmail, Outlook, your web host\'s email server, etc.) and its credentials.\n\n```python\nimport smtplib\nfrom email.mime.text import MIMEText\nfrom email.mime.multipart import MIMEMultipart\n\n# --- Email Configuration ---\nsender_email = "your_email@example.com"  # Replace with your actual email address\nsender_password = "your_app_password" # Replace with your email password or app password\nreceiver_email = "john@test.c

In [31]:
# Step 4 - Editing 
config = {"configurable":{"thread_id":"test-edit"}}

result = agent.invoke(
    {"messages":[HumanMessage(content="Send Email to wrong@test.com with subject 'Hello' and the body 'How are you?  '")]},config = config
)

In [32]:
result

{'messages': [HumanMessage(content="Send Email to wrong@test.com with subject 'Hello' and the body 'How are you?  '", additional_kwargs={}, response_metadata={}, id='8ec58b98-1661-44ac-aa74-5e371d20000d'),
  AIMessage(content="Okay, I will send an email to wrong@test.com with the subject 'Hello' and the body 'How are you?  '.", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a060f4-c70b-7702-b2ea-304a2cab577d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 89, 'total_tokens': 113, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 60}}),
  HumanMessage(content="Send Email to wrong@test.com with subject 'Hello' and the body 'How are you?  '", additional_kwargs={}, response_metadata={}, id='6fa8a368-4462-470d-a87b-ee91916e27d6'),
  AIMessage(content="Okay, I am sending another email to wrong@